# Lesson 3

The chain is the most important building block - usually combines a LLM together with a prompt. You can put a bunch of these building blocks together to carry out a sequence of operations on your text or on your other data.

In [9]:
# load environment variables
import warnings
warnings.filterwarnings('ignore')

import os
import pandas as pd
from langchain_openai import ChatOpenAI
from langchain.prompts import ChatPromptTemplate
from langchain.chains import LLMChain

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file

# set llm model
llm_model = "gpt-3.5-turbo"

In [8]:
# load data
df = pd.read_csv('../.data/product_reviews_L3.csv')
df.head()

,Product,Review
0,Queen Size Sheet Set,I ordered a king size set. My only criticism w...
1,Waterproof Phone Pouch,"I loved the waterproof sac, although the openi..."
2,Luxury Air Mattress,This mattress had a small hole in the top of i...
3,Pillows Insert,This is the best throw pillow fillers on Amazo...
4,Milk Frother Handheld,I loved this product. But they only seem to la...


# Creating an LLM Chain

In [10]:
# initalise LLM (setting high temperature for some fun descriptions)
llm = ChatOpenAI(temperature=0.9, model=llm_model)

In [11]:
# initalise prompt template
prompt = ChatPromptTemplate.from_template(
    "What is the best name to describe \
    a company that makes {product}?"
)

In [14]:
# combining the LLM instance and prompt template to create a chain, making it simplier to ask the llm something
chain = LLMChain(llm=llm, prompt=prompt)

# setting product variable
product = "Queen Size Sheet Set"

# this creates the prompt and then uses it to prompt the LLM
chain.run(product)

/var/folders/7j/npjz00mn76s2k495vh_q9js00000gn/T/ipykernel_7375/3192387590.py:8: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  chain.run(product)


'"Royal Slumber Linens"'

# Sequential chains
Sequential chain is another type of chains. The idea is to combine multiple chains where the output of one chain is the input for the next chain.

There are two types of sequential chains:
* SimpleSequentialChain: isngle input/output
* SequentialChain: multiple inputs/outputs

# SimpleSequentialChain
Works well for when you need single inputs and single outputs.

In [15]:
from langchain.chains import SimpleSequentialChain

In [17]:
# defining model
llm = ChatOpenAI(temperature=0.9, model=llm_model)

# prompt template 1
first_prompt = ChatPromptTemplate.from_template(
    "What is the best name to describe \
    a company that makes {product}?"
)

# Chain 1
chain_one = LLMChain(llm=llm, prompt=first_prompt)

In [18]:
# prompt template 2
second_prompt = ChatPromptTemplate.from_template(
    "Write a 20 words description for the following \
    company:{company_name}"
)
# chain 2
chain_two = LLMChain(llm=llm, prompt=second_prompt)

In [20]:
# creating sequential chain
overall_simple_chain = SimpleSequentialChain(chains=[chain_one, chain_two],
                                             verbose=True
                                            )

In [22]:
# run the chain over any product (using example product variable from earlier)
overall_simple_chain.run(product)



> Entering new SimpleSequentialChain chain...
"Royal Dream Linens"
Royal Dream Linens offers luxurious bedding and home decor products fit for royalty, making every night feel like a dream.

> Finished chain.


'Royal Dream Linens offers luxurious bedding and home decor products fit for royalty, making every night feel like a dream.'

# SequentialChain

Works well for multiple inputs and outputs (input and output keys need to be precise)

In [26]:
from langchain.chains import SequentialChain

In [32]:
# define model
llm = ChatOpenAI(temperature=0.9, model=llm_model)

# prompt template 1: translate to english
first_prompt = ChatPromptTemplate.from_template(
    "Translate the following review to french:"
    "\n\n{Review}"
)
# chain 1: input= Review and output= English_Review
chain_one = LLMChain(llm=llm, prompt=first_prompt, 
                     output_key="French_Review"
                    )

# prompt template 2
second_prompt = ChatPromptTemplate.from_template(
    "Can you summarize the following review in 1 sentence:"
    "\n\n{French_Review}"
)
# chain 2: input= English_Review and output= summary
chain_two = LLMChain(llm=llm, prompt=second_prompt, 
                     output_key="summary"
                    )

# prompt template 3
third_prompt = ChatPromptTemplate.from_template(
    "What language is the following review:\n\n{Review}"
)
# chain 3: input= Review and output= language
chain_three = LLMChain(llm=llm, prompt=third_prompt,
                       output_key="language"
                      )

# prompt template 4: follow up message
fourth_prompt = ChatPromptTemplate.from_template(
    "Write a follow up response to the following "
    "summary in the specified language:"
    "\n\nSummary: {summary}\n\nLanguage: {language}"
)
# chain 4: input= summary, language and output= followup_message
chain_four = LLMChain(llm=llm, prompt=fourth_prompt,
                      output_key="followup_message"
                     )


In [34]:
# overall_chain: input= Review 
# and output= English_Review,summary, followup_message
overall_chain = SequentialChain(
    chains=[chain_one, chain_two, chain_three, chain_four],
    input_variables=["Review"],
    output_variables=["French_Review", "summary","followup_message"],
    verbose=True
)

In [ ]:
# select the review to input
review = df.Review[5]

# call the chain and see outputs
overall_chain(review)



> Entering new SequentialChain chain...

> Finished chain.


{'Review': 'Great for smoothies, but battery life could be better.',
 'French_Review': "Idéal pour les smoothies, mais l'autonomie de la batterie pourrait être meilleure.",
 'summary': 'Great for making smoothies, but the battery life could be better.',
 'followup_message': 'Thank you for your feedback. We appreciate your input and understand your concerns about the battery life. We will take this into consideration for future improvements. In the meantime, we hope you continue to enjoy using our product for making smoothies.'}

# Router Chain
A router chain can be used to decide which subchain to pass the output (of the router chain) to.

Example below shows how you might route between different chains based on the subject.


In [37]:
# defining all the prompt templates

physics_template = """You are a very smart physics professor. \
You are great at answering questions about physics in a concise\
and easy to understand manner. \
When you don't know the answer to a question you admit\
that you don't know.

Here is a question:
{input}"""


math_template = """You are a very good mathematician. \
You are great at answering math questions. \
You are so good because you are able to break down \
hard problems into their component parts, 
answer the component parts, and then put them together\
to answer the broader question.

Here is a question:
{input}"""

history_template = """You are a very good historian. \
You have an excellent knowledge of and understanding of people,\
events and contexts from a range of historical periods. \
You have the ability to think, reflect, debate, discuss and \
evaluate the past. You have a respect for historical evidence\
and the ability to make use of it to support your explanations \
and judgements.

Here is a question:
{input}"""


computerscience_template = """ You are a successful computer scientist.\
You have a passion for creativity, collaboration,\
forward-thinking, confidence, strong problem-solving capabilities,\
understanding of theories and algorithms, and excellent communication \
skills. You are great at answering coding questions. \
You are so good because you know how to solve a problem by \
describing the solution in imperative steps \
that a machine can easily interpret and you know how to \
choose a solution that has a good balance between \
time complexity and space complexity. 

Here is a question:
{input}"""

In [ ]:
# provide more information about the prompt templates (to be used bny the router chain)

prompt_infos = [
    {
        "name": "physics", 
        "description": "Good for answering questions about physics", 
        "prompt_template": physics_template
    },
    {
        "name": "math", 
        "description": "Good for answering math questions", 
        "prompt_template": math_template
    },
    {
        "name": "History", 
        "description": "Good for answering history questions", 
        "prompt_template": history_template
    },
    {
        "name": "computer science", 
        "description": "Good for answering computer science questions", 
        "prompt_template": computerscience_template
    }
]

In [40]:
# used for routine between multiple prompt templates
from langchain.chains.router import MultiPromptChain 

# LLM Router Chain uses an LLM to route between different subchains
# RouterOutputParser parses LLM output into dictionary (can be used to determine which chain to use and what the input should be)
from langchain.chains.router.llm_router import LLMRouterChain,RouterOutputParser

from langchain.prompts import PromptTemplate

In [41]:
# define LLM
llm = ChatOpenAI(temperature=0, model=llm_model)

In [42]:
# create destination chains

destination_chains = {}
for p_info in prompt_infos:
    name = p_info["name"]
    prompt_template = p_info["prompt_template"]
    prompt = ChatPromptTemplate.from_template(template=prompt_template)
    chain = LLMChain(llm=llm, prompt=prompt) # each destination chain is an LLM chain
    destination_chains[name] = chain  
    
destinations = [f"{p['name']}: {p['description']}" for p in prompt_infos]
destinations_str = "\n".join(destinations)

In [52]:
# create default chain which is a generic call to the LLM (this chain is called when the router chain can't decide which of the subchains to use)
default_prompt = ChatPromptTemplate.from_template("{input}")
default_chain = LLMChain(llm=llm, prompt=default_prompt)

In [ ]:
# define the template that is used by the LLM to route between the different chains
MULTI_PROMPT_ROUTER_TEMPLATE = """Given a raw text input to a \
language model select the model prompt best suited for the input. \
You will be given the names of the available prompts and a \
description of what the prompt is best suited for. \
You may also revise the original input if you think that revising\
it will ultimately lead to a better response from the language model.

<< FORMATTING >>
Return a markdown code snippet with a JSON object formatted to look like:
```json
{{{{
    "destination": string \ "DEFAULT" or name of the prompt to use in {destinations}
    "next_inputs": string \ a potentially modified version of the original input
}}}}
```

REMEMBER: The value of “destination” MUST match one of \
the candidate prompts listed below.\
If “destination” does not fit any of the specified prompts, set it to “DEFAULT.”
REMEMBER: "next_inputs" can just be the original input \
if you don't think any modifications are needed.

<< CANDIDATE PROMPTS >>
{destinations}

<< INPUT >>
{{input}}

<< OUTPUT (remember to include the ```json)>>"""

In [46]:
# create prompt template
router_template = MULTI_PROMPT_ROUTER_TEMPLATE.format(
    destinations=destinations_str
)

# create the prompt
router_prompt = PromptTemplate(
    template=router_template,
    input_variables=["input"],
    output_parser=RouterOutputParser(),
)

router_chain = LLMRouterChain.from_llm(llm, router_prompt)

In [48]:
# create overall chain made up of lots of other chains
chain = MultiPromptChain(router_chain=router_chain, # router chain
                         destination_chains=destination_chains, # destination chains
                         default_chain=default_chain, # default chain
                         verbose=True
                        )

In [49]:
chain.run("What is black body radiation?")



> Entering new MultiPromptChain chain...
physics: {'input': 'What is black body radiation?'}
> Finished chain.


"Black body radiation is the electromagnetic radiation emitted by a perfect absorber and emitter of radiation, known as a black body. A black body absorbs all radiation that falls on it and emits radiation at all wavelengths. The spectrum of black body radiation is continuous and depends only on the temperature of the black body. This phenomenon is described by Planck's law, which states that the intensity of radiation emitted by a black body at a given wavelength is proportional to the temperature of the body and the wavelength raised to the fifth power."

In [50]:
chain.run("what is 2 + 2")



> Entering new MultiPromptChain chain...
math: {'input': 'what is 2 + 2'}
> Finished chain.


'The answer to 2 + 2 is 4.'

In [51]:
chain.run("Why does every cell in our body contain DNA?")



> Entering new MultiPromptChain chain...
None: {'input': 'Why does every cell in our body contain DNA?'}
> Finished chain.


'Every cell in our body contains DNA because DNA carries the genetic information that determines the characteristics and functions of an organism. DNA contains the instructions for building and maintaining an organism, including the proteins that are essential for cell function and structure. This genetic information is passed down from parent to offspring and is essential for the growth, development, and functioning of all cells in the body. Having DNA in every cell ensures that the genetic information is preserved and can be used to carry out the necessary processes for life.'